# Day 57 — NumPy for Analysts
**Month 3 | Week 6 | Arrays · Broadcasting · Vectorised Math · np.where · Practical Use Cases**

---

> **Real-world framing:**
>
> Every Pandas DataFrame is built on NumPy arrays underneath.
> Every Scikit-learn model inputs and outputs NumPy arrays.
> Every time you write `.values`, `.to_numpy()`, or a vectorised operation — that's NumPy.
>
> A data analyst who doesn't understand NumPy works around the engine.
> One who does works *with* it — writing code that is 10–100× faster, cleaner,
> and directly compatible with ML pipelines.
>
> Today you learn the five NumPy skills that appear in every real analytics workflow:
> array creation, vectorised math, broadcasting, `np.where`, and practical
> interop with Pandas.

---

**Skills used today:** Python basics, Pandas (all Month 3 prior days)  
**New today:** `np.array` · `np.arange/linspace` · broadcasting · vectorised ops ·
`np.where` · `np.select` · `np.percentile` · `np.random` · Pandas ↔ NumPy interop

**Total: 80 pts + 10★ bonus**

---


---
## 📦 Section 1 — Raw Data (Read Only — Never Modify)

Same ShopEase dataset (200 records, seed=7). Plus a small NumPy-native array dataset for Sections A–B.

In [1]:
# ── RAW DATA — DO NOT MODIFY BELOW THIS CELL ──────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── ShopEase DataFrame (familiar) ──────────────────────────────────────────
rng_np = np.random.default_rng(seed=7)
n = 200
regions    = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books']
segments   = ['Regular', 'Premium', 'VIP']
statuses   = ['Delivered', 'Returned', 'Pending']

raw = pd.DataFrame({
    'order_id'    : [f'ORD{1000+i}' for i in range(n)],
    'order_date'  : pd.date_range('2024-01-01', periods=n, freq='2D'),
    'region'      : rng_np.choice(regions,    n, p=[0.30, 0.25, 0.25, 0.20]),
    'category'    : rng_np.choice(categories, n, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'units'       : rng_np.integers(1, 20, n),
    'unit_price'  : rng_np.uniform(50, 500, n).round(2),
    'discount_pct': rng_np.choice([0, 5, 10, 15, 20], n, p=[0.4, 0.25, 0.20, 0.10, 0.05]),
    'segment'     : rng_np.choice(segments,   n, p=[0.50, 0.35, 0.15]),
    'status'      : rng_np.choice(statuses,   n, p=[0.70, 0.20, 0.10]),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)
raw['rev_tier'] = pd.cut(raw['revenue'], bins=[0, 1000, 4000, np.inf],
                          labels=['Low Value', 'Mid Value', 'High Value'])
df = raw.copy()

# ── NumPy-native arrays (for Tasks A and B) ────────────────────────────────
# Simulated monthly sales figures for 4 regions × 6 months (Jan–Jun 2024)
monthly_sales = np.array([
    [12400, 15300, 11800, 16700, 14200, 18900],   # North
    [10200, 13100, 12500, 11900, 15800, 14300],   # South
    [16800, 14500, 17200, 19100, 16300, 21000],   # East
    [ 9800, 11200, 10600, 12800, 11500, 13700],   # West
], dtype=float)

region_names = np.array(['North', 'South', 'East', 'West'])
month_names  = np.array(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'])

print(f"ShopEase df: {df.shape}")
print(f"monthly_sales shape: {monthly_sales.shape}  dtype: {monthly_sales.dtype}")
print()
print("monthly_sales array:")
print(monthly_sales)


ShopEase df: (200, 11)
monthly_sales shape: (4, 6)  dtype: float64

monthly_sales array:
[[12400. 15300. 11800. 16700. 14200. 18900.]
 [10200. 13100. 12500. 11900. 15800. 14300.]
 [16800. 14500. 17200. 19100. 16300. 21000.]
 [ 9800. 11200. 10600. 12800. 11500. 13700.]]


---
## 📖 Section 2 — Concept Notes

### 1. NumPy Arrays — The Foundation

A NumPy array is a fixed-type, fixed-size, contiguous block of memory.
That's what makes it fast — no Python overhead per element.

```python
import numpy as np

# 1D array
arr = np.array([1, 2, 3, 4, 5])

# 2D array (matrix) — shape (rows, cols)
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])    # shape (2, 3)

# Key attributes
arr.shape    # (5,)
arr.dtype    # int64 / float64
arr.ndim     # 1 (dimensions)
arr.size     # 5 (total elements)
```

**Array creation shortcuts:**
```python
np.zeros((3, 4))          # 3×4 array of 0.0
np.ones((2, 5))           # 2×5 array of 1.0
np.arange(0, 10, 2)       # [0, 2, 4, 6, 8]
np.linspace(0, 1, 5)      # [0.0, 0.25, 0.5, 0.75, 1.0]
np.eye(3)                  # 3×3 identity matrix
```

---

### 2. Vectorised Operations — No Loops Needed

NumPy applies operations to every element simultaneously at C speed.

```python
prices = np.array([100, 200, 300, 400])
discount = 0.10

# BAD — Python loop
discounted = []
for p in prices:
    discounted.append(p * (1 - discount))

# GOOD — vectorised (same result, 100× faster)
discounted = prices * (1 - discount)
```

**Element-wise math:**
```python
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
a + b       # [11, 22, 33]
a * b       # [10, 40, 90]
a ** 2      # [1, 4, 9]
np.sqrt(b)  # [3.16, 4.47, 5.47]
np.log(b)   # natural log element-wise
```

---

### 3. Broadcasting — Shapes That Don't Match, But Work

Broadcasting lets NumPy apply operations across arrays of different shapes
by automatically expanding the smaller array.

```python
monthly = np.array([[100, 200, 300],   # shape (2, 3)
                    [400, 500, 600]])

targets = np.array([150, 250, 350])    # shape (3,)
# NumPy broadcasts targets across rows
above_target = monthly > targets       # compares each column to its target

# Common pattern: subtract column mean (normalise)
monthly - monthly.mean(axis=0)        # axis=0 = along rows (per column)
monthly - monthly.mean(axis=1, keepdims=True)  # axis=1 = along cols (per row)
```

**axis cheatsheet for 2D arrays:**
| `axis=` | Direction | Result |
|---------|-----------|--------|
| `0` | Down rows | One value per column |
| `1` | Across cols | One value per row |
| `None` | All elements | Single scalar |

---

### 4. np.where — Vectorised if/else

`np.where(condition, value_if_true, value_if_false)`

```python
revenue = np.array([500, 2000, 6000, 800, 3500])

# Single condition
tier = np.where(revenue > 4000, 'High', 'Other')

# Chained conditions (nested np.where)
tier = np.where(revenue > 4000, 'High',
       np.where(revenue > 1000, 'Mid', 'Low'))
```

For 3+ conditions, prefer `np.select` — cleaner:
```python
conditions = [revenue > 4000, revenue > 1000]
choices    = ['High Value', 'Mid Value']
tier = np.select(conditions, choices, default='Low Value')
```

---

### 5. Pandas ↔ NumPy Interop

```python
# DataFrame column → NumPy array
arr = df['revenue'].to_numpy()       # preferred
arr = df['revenue'].values           # also works

# NumPy array → DataFrame column
df['scaled'] = arr / arr.max()

# Apply NumPy function to column
np.percentile(df['revenue'], [25, 50, 75])   # quartiles
np.clip(df['revenue'], a_min=100, a_max=5000)  # cap values
np.log1p(df['revenue'])                        # log(1+x) — handles zeros safely
```

**Why `.to_numpy()` over `.values`?**  
`.to_numpy()` handles extension types (nullable integers, categoricals) correctly.
`.values` can return object arrays for categoricals. Use `.to_numpy()` in production.

---

### Common Mistakes → Fixes

| Mistake | Fix |
|---------|-----|
| `arr[0, :]` vs `arr[0]` confusion | Both work for 2D rows; `:` is explicit |
| `axis=0` vs `axis=1` reversed | axis=0 collapses rows → result per column |
| Using loop instead of vectorised op | Rewrite as `arr * scalar` or `np.where` |
| `np.where` with 3 tiers in one call | Use `np.select` for 3+ conditions |
| Forgetting `keepdims=True` in broadcasting | Without it, shapes mismatch in subtraction |


---
## ✏️ Section 3 — Practice Tasks

Total = **80 pts + 10★ bonus**. Never modify raw data.

---
### Task A — Array Operations on monthly_sales (20 pts)

Work with the `monthly_sales` array (shape 4×6) and `region_names` / `month_names`.

#### A1 — Basic Stats per Region (10 pts)
Without converting to Pandas, compute for each region using NumPy axis operations:

| Metric | Function |
|--------|----------|
| Total sales (sum across 6 months) | `axis=1` |
| Monthly average | `axis=1` |
| Best month value | `axis=1` |
| Worst month value | `axis=1` |

Print a clean table showing all four metrics for each region.
Use a loop over `region_names` to print — no Pandas here.

Then: which region has the highest total? Which has the highest monthly average?
State both as printed f-strings using computed values (not hardcoded).


In [2]:
# A1 — Basic Stats per Region (10 pts)
# Compute: totals, averages, best_month, worst_month — all along axis=1
# Loop over region_names to print a clean table
# Then print: which region leads in total? which in average?
totals    = monthly_sales.sum(axis=1)
averages  = monthly_sales.mean(axis=1)
best_mon  = monthly_sales.max(axis=1)
worst_mon = monthly_sales.min(axis=1)

print(f"{'Region':<8} {'Total':>10} {'Avg':>10} {'Best':>8} {'Worst':>8}")
print("-" * 46)
for i, r in enumerate(region_names):
    print(f"{r:<8} {totals[i]:>10,.0f} {averages[i]:>10,.1f} {best_mon[i]:>8,.0f} {worst_mon[i]:>8,.0f}")

top_total = region_names[np.argmax(totals)]
top_avg   = region_names[np.argmax(averages)]
print(f"\nHighest total:   {top_total} at ₹{totals.max():,.0f}")
print(f"Highest average: {top_avg} at ₹{averages.max():,.1f}")



Region        Total        Avg     Best    Worst
----------------------------------------------
North        89,300   14,883.3   18,900   11,800
South        77,800   12,966.7   15,800   10,200
East        104,900   17,483.3   21,000   14,500
West         69,600   11,600.0   13,700    9,800

Highest total:   East at ₹104,900
Highest average: East at ₹17,483.3


#### A2 — Broadcasting: vs Target + Growth Rate (10 pts)
**Part 1 — Above target:**
Define a monthly target array:
```python
targets = np.array([12000, 13000, 12000, 14000, 14000, 16000])
```
Using broadcasting (one line, no loop), create a boolean array `above_target`
where `True` = region × month combination exceeded target.

Print:
- The `above_target` boolean array (4×6)
- Count of True values per region (how many months did each region beat its target?)
- Which region beat its target most often?

**Part 2 — Month-on-month growth rate:**
Compute the month-on-month percentage growth for each region:
```
growth[region, month] = (sales[month] - sales[month-1]) / sales[month-1] * 100
```
This produces a 4×5 array (5 growth values from 6 months).
Use array slicing: `monthly_sales[:, 1:]` and `monthly_sales[:, :-1]`.
Round to 1 dp.

Print the growth array. Then print which (region, month-transition) had
the largest single-month growth — use `np.unravel_index(np.argmax(...), shape)`.


In [3]:
# A2 — Broadcasting + Growth Rate (10 pts)
# Part 1:
# targets = np.array([12000, 13000, 12000, 14000, 14000, 16000])
# above_target = monthly_sales > targets   ← one line broadcasting
# Print array, count per region, best region
# Part 2:
# growth = (monthly_sales[:, 1:] - monthly_sales[:, :-1]) / monthly_sales[:, :-1] * 100
# Round to 1dp, print, find argmax with np.unravel_index
targets = np.array([12000, 13000, 12000, 14000, 14000, 16000])
above_target = monthly_sales > targets
print("Above target (bool array):")
print(above_target)
beats = above_target.sum(axis=1)
print("\nMonths beating target per region:")
for i, r in enumerate(region_names):
    print(f"  {r}: {beats[i]}/6")
print(f"Most consistent region: {region_names[np.argmax(beats)]}")

growth = ((monthly_sales[:, 1:] - monthly_sales[:, :-1]) / monthly_sales[:, :-1] * 100).round(1)
print("\nMonth-on-month growth (%):")
print(growth)
flat_idx = np.argmax(growth)
r_i, m_i = np.unravel_index(flat_idx, growth.shape)
month_pairs = ['Jan→Feb','Feb→Mar','Mar→Apr','Apr→May','May→Jun']
print(f"Largest growth: {region_names[r_i]} in {month_pairs[m_i]} at {growth[r_i, m_i]}%")



Above target (bool array):
[[ True  True False  True  True  True]
 [False  True  True False  True False]
 [ True  True  True  True  True  True]
 [False False False False False False]]

Months beating target per region:
  North: 5/6
  South: 3/6
  East: 6/6
  West: 0/6
Most consistent region: East

Month-on-month growth (%):
[[ 23.4 -22.9  41.5 -15.   33.1]
 [ 28.4  -4.6  -4.8  32.8  -9.5]
 [-13.7  18.6  11.  -14.7  28.8]
 [ 14.3  -5.4  20.8 -10.2  19.1]]
Largest growth: North in Mar→Apr at 41.5%


---
### Task B — np.where and np.select on ShopEase (20 pts)

Work on the ShopEase `df`. Extract NumPy arrays from columns using `.to_numpy()`.

#### B1 — Vectorised Discount Flag (8 pts)
Extract `discount_pct` as a NumPy array.

Using **`np.where`** (no loops, no apply), create array `discount_flag`:
- `discount_pct == 0` → `'No Discount'`
- `discount_pct <= 10` → `'Low Discount'`  
- `discount_pct > 10` → `'High Discount'`

> **Tip:** Two nested `np.where` calls, or use `np.select`.

Add `discount_flag` back to `df` as a new column.
Print `df['discount_flag'].value_counts()`.


In [4]:
# B1 — Vectorised Discount Flag (8 pts)
# discount_arr = df['discount_pct'].to_numpy()
# discount_flag = np.where(... np.where(...))   OR use np.select
# df['discount_flag'] = discount_flag
# Print value_counts
discount_arr = df['discount_pct'].to_numpy()
conditions_b1 = [discount_arr == 0, discount_arr <= 10]
choices_b1    = ['No Discount', 'Low Discount']
df['discount_flag'] = np.select(conditions_b1, choices_b1, default='High Discount')
print(df['discount_flag'].value_counts())



discount_flag
No Discount      92
Low Discount     82
High Discount    26
Name: count, dtype: int64


#### B2 — np.select for Revenue Scoring (12 pts)
Extract `revenue` as a NumPy array.

Compute the 25th and 75th percentiles of revenue using `np.percentile`.
Print them.

Then use **`np.select`** to create array `rev_score`:
- revenue ≥ 75th percentile → `3` (High)
- revenue ≥ 25th percentile (but < 75th) → `2` (Mid)
- revenue < 25th percentile → `1` (Low)

Add `rev_score` to `df`. Print:
- `df['rev_score'].value_counts().sort_index()`
- Mean revenue for each score group:
  `df.groupby('rev_score')['revenue'].mean().round(2)`

Then write **one NRA sentence**: what does the rev_score distribution tell
the sales team about where to focus retention efforts?


In [5]:
# B2 — np.select Revenue Scoring (12 pts)
# revenue_arr = df['revenue'].to_numpy()
# p25, p75 = np.percentile(revenue_arr, [25, 75])
# conditions and choices for np.select
# df['rev_score'] = rev_score
# Print value_counts and groupby mean
# NRA in markdown below
revenue_arr = df['revenue'].to_numpy()
p25, p75 = np.percentile(revenue_arr, [25, 75])
print(f"P25: ₹{p25:.2f}  |  P75: ₹{p75:.2f}")

conditions_b2 = [revenue_arr >= p75, revenue_arr >= p25]
choices_b2    = [3, 2]
df['rev_score'] = np.select(conditions_b2, choices_b2, default=1)
print(df['rev_score'].value_counts().sort_index())
print(df.groupby('rev_score')['revenue'].mean().round(2))



P25: ₹972.90  |  P75: ₹3747.73
rev_score
1     50
2    100
3     50
Name: count, dtype: int64
rev_score
1     565.00
2    2226.13
3    5374.70
Name: revenue, dtype: float64


**N:** 50 orders (25%) are high‑revenue (score 3) with a mean of ₹5,374.70, 100 orders (50%) mid‑range (score 2, mean ₹2,226.13), and 50 orders (25%) low (score 1, mean ₹565.00).

**R:** The revenue base is highly skewed – the high‑score segment contributes roughly 50% of total revenue despite being only a quarter of orders. The mid‑score group represents the largest share and a substantial revenue pool.

**A:** Sales should prioritise up‑selling and loyalty initiatives for the mid‑score segment (score 2) to nudge them into the high‑score bracket, which would dramatically lift overall revenue.


---
### Task C — Vectorised Calculations (20 pts)

These are the types of operations that appear in real ML preprocessing pipelines.
All work on NumPy arrays extracted from `df`. No loops allowed.

#### C1 — Revenue Normalisation (10 pts)
Extract `revenue` as `rev_arr = df['revenue'].to_numpy()`.

Compute three transformations (one line each):

**Min-Max scaling** (normalise to 0–1 range):
```
scaled = (rev_arr - rev_arr.min()) / (rev_arr.max() - rev_arr.min())
```

**Z-score standardisation** (mean=0, std=1):
```
z_score = (rev_arr - rev_arr.mean()) / rev_arr.std()
```

**Log transformation** (reduce skew — use `np.log1p`):
```
log_rev = np.log1p(rev_arr)
```

Add all three as columns to `df`:
`'rev_scaled'`, `'rev_zscore'`, `'rev_log'`

Print:
```
Min-Max: min=X, max=X  (should be exactly 0.0 and 1.0)
Z-score: mean=X, std=X  (should be ~0.0 and ~1.0)
Log: min=X, max=X
```
Round all printed values to 4 dp.


In [6]:
# C1 — Revenue Normalisation (10 pts)
# rev_arr = df['revenue'].to_numpy()
# Compute: scaled, z_score, log_rev — each in ONE line
# Add to df as: rev_scaled, rev_zscore, rev_log
# Print verification stats
rev_arr = df['revenue'].to_numpy()
df['rev_scaled'] = (rev_arr - rev_arr.min()) / (rev_arr.max() - rev_arr.min())
df['rev_zscore'] = (rev_arr - rev_arr.mean()) / rev_arr.std()
df['rev_log']    = np.log1p(rev_arr)
print(f"Min-Max: min={df['rev_scaled'].min():.4f}, max={df['rev_scaled'].max():.4f}")
print(f"Z-score: mean={df['rev_zscore'].mean():.4f}, std={df['rev_zscore'].std():.4f}")
print(f"Log:     min={df['rev_log'].min():.4f}, max={df['rev_log'].max():.4f}")



Min-Max: min=0.0000, max=1.0000
Z-score: mean=0.0000, std=1.0025
Log:     min=4.0799, max=9.0387


#### C2 — Vectorised Revenue Projection (10 pts)
Create a NumPy array `growth_rates` of 6 monthly growth rates for projection:
```python
growth_rates = np.array([0.02, 0.03, 0.025, 0.04, 0.035, 0.03])
```

Take `base_revenue = df['revenue'].to_numpy()` (shape: (200,)).

Use **broadcasting** to compute projected revenue for each of the 6 months:
- Month 1: `base_revenue * (1 + growth_rates[0])`
- Month 2: `base_revenue * (1 + growth_rates[1])`
- ... and so on for all 6

Do this in **one operation** using broadcasting (no loop):
```python
# Hint: reshape base_revenue to (200, 1) so it broadcasts against (6,)
projections = base_revenue.reshape(-1, 1) * (1 + growth_rates)
```

`projections` should have shape **(200, 6)**.

Print:
- `projections.shape`
- Mean projected revenue per month (axis=0): round to 2dp
- Which month has the highest mean projected revenue?
- Total projected revenue across all orders for month 6: round to 2dp


In [7]:
# C2 — Vectorised Revenue Projection (10 pts)
# growth_rates = np.array([0.02, 0.03, 0.025, 0.04, 0.035, 0.03])
# base_revenue = df['revenue'].to_numpy()
# projections = base_revenue.reshape(-1, 1) * (1 + growth_rates)
# Print: shape, mean per month, highest month, total month 6
growth_rates = np.array([0.02, 0.03, 0.025, 0.04, 0.035, 0.03])
base_revenue = df['revenue'].to_numpy()
projections  = base_revenue.reshape(-1, 1) * (1 + growth_rates)
print(f"Shape: {projections.shape}")
monthly_means = projections.mean(axis=0).round(2)
print(f"Mean projected revenue per month: {monthly_means}")
best_month_idx = np.argmax(monthly_means)
print(f"Highest mean month: Month {best_month_idx+1} at ₹{monthly_means[best_month_idx]:,.2f}")
print(f"Total projected revenue Month 6: ₹{projections[:, 5].sum():,.2f}")



Shape: (200, 6)
Mean projected revenue per month: [2649.95 2675.93 2662.94 2701.91 2688.92 2675.93]
Highest mean month: Month 4 at ₹2,701.91
Total projected revenue Month 6: ₹535,186.17


---
### Task D — NumPy + Pandas Integration (20 pts)

Combine NumPy array operations with Pandas for the kind of workflow that
appears in real data pipelines and Scikit-learn preprocessing.

#### D1 — Clipping and Outlier Detection (10 pts)
Extract `revenue` as `rev_arr`.

**Step 1 — IQR outlier detection:**
Compute Q1, Q3, and IQR using `np.percentile`.
Compute lower bound = Q1 − 1.5 × IQR and upper bound = Q3 + 1.5 × IQR.
Use `np.where` to flag outliers: `1` if outside bounds, `0` otherwise.
Add column `'is_outlier'` to `df`.

**Step 2 — Clip outliers:**
Use `np.clip(rev_arr, lower_bound, upper_bound)` to cap values.
Add column `'revenue_clipped'` to `df`.

Print:
- How many outliers were detected (sum of `is_outlier`)
- Original revenue stats: mean, std, max
- Clipped revenue stats: mean, std, max
- One sentence: what changed after clipping and why that matters for ML models?


In [8]:
# D1 — IQR Outlier Detection + Clipping (10 pts)
# rev_arr = df['revenue'].to_numpy()
# q1, q3 = np.percentile(rev_arr, [25, 75])
# iqr = q3 - q1
# lower, upper bounds
# is_outlier flag with np.where
# revenue_clipped with np.clip
# Print comparison stats + interpretation
rev_arr = df['revenue'].to_numpy()
q1, q3 = np.percentile(rev_arr, [25, 75])
iqr = q3 - q1
lower_b = q1 - 1.5 * iqr
upper_b = q3 + 1.5 * iqr
df['is_outlier']      = np.where((rev_arr < lower_b) | (rev_arr > upper_b), 1, 0)
df['revenue_clipped'] = np.clip(rev_arr, lower_b, upper_b)

print(f"Outliers detected: {df['is_outlier'].sum()}")
print(f"Original  — mean: {rev_arr.mean():.2f}, std: {rev_arr.std():.2f}, max: {rev_arr.max():.2f}")
print(f"Clipped   — mean: {df['revenue_clipped'].mean():.2f}, std: {df['revenue_clipped'].std():.2f}, max: {df['revenue_clipped'].max():.2f}")
print("Clipping reduced the mean and max by capping 3 outliers; this prevents extreme values from skewing ML model coefficients, leading to more robust and generalisable predictions.")


Outliers detected: 3
Original  — mean: 2597.99, std: 1952.79, max: 8421.97
Clipped   — mean: 2590.64, std: 1936.59, max: 7909.98
Clipping reduced the mean and max by capping 3 outliers; this prevents extreme values from skewing ML model coefficients, leading to more robust and generalisable predictions.


#### D2 — Feature Matrix for ML (10 pts)
A Scikit-learn model expects a numeric feature matrix (NumPy array).
Build one from `df` using the columns you've created today.

**Step 1:** Select these numeric columns from `df`:
`'units'`, `'unit_price'`, `'discount_pct'`, `'rev_scaled'`, `'rev_zscore'`, `'rev_log'`, `'rev_score'`

Convert to a NumPy array `X`:
```python
feature_cols = ['units', 'unit_price', 'discount_pct', 'rev_scaled', 'rev_zscore', 'rev_log', 'rev_score']
X = df[feature_cols].to_numpy()
```

**Step 2:** Print:
- `X.shape`  
- `X.dtype`
- Column-wise mean (one value per feature): round to 3dp
- Column-wise std: round to 3dp

**Step 3:** Check for any NaN or Inf values:
```python
print("NaN count:", np.isnan(X).sum())
print("Inf count:", np.isinf(X).sum())
```

**Step 4:** Write one NRA: what does having a clean (no NaN, no Inf) feature matrix
mean for the downstream ML workflow?


In [9]:
# D2 — Feature Matrix for ML (10 pts)
# feature_cols list
# X = df[feature_cols].to_numpy()
# Print shape, dtype, column means, column stds
# Check np.isnan and np.isinf
# NRA in markdown below
feature_cols = ['units', 'unit_price', 'discount_pct', 'rev_scaled', 'rev_zscore', 'rev_log', 'rev_score']
X = df[feature_cols].to_numpy()
print(f"X.shape: {X.shape}")
print(f"X.dtype: {X.dtype}")
print(f"Column means: {X.mean(axis=0).round(3)}")
print(f"Column stds:  {X.std(axis=0).round(3)}")
print(f"NaN count: {np.isnan(X).sum()}")
print(f"Inf count: {np.isinf(X).sum()}")



X.shape: (200, 7)
X.dtype: float64
Column means: [ 10.1   266.472   5.05    0.304   0.      7.489   2.   ]
Column stds:  [  5.285 127.013   5.852   0.233   1.      0.99    0.707]
NaN count: 0
Inf count: 0


**N:** The feature matrix `X` has shape (200, 7), dtype float64, with 0 NaN and 0 Inf values.

**R:** A clean, complete numeric matrix means no missing or infinite inputs that would break ML algorithms – every row is ready to be used for training or inference without imputation or error‑handling.

**A:** This matrix can be fed directly into a Scikit‑learn pipeline (e.g., `StandardScaler` + `LogisticRegression`) without additional cleaning, saving development time and reducing the risk of silent failures.

---
## 📊 Section 4 — Scoring Rubric

| Task | Sub-Task | Pts | What Is Checked |
|------|----------|-----|-----------------|
| **A — Arrays** | A1 Stats per region | 10 | axis=1 for all 4 metrics, loop prints clean table, top region identified dynamically |
| | A2 Broadcasting + Growth | 10 | `>` broadcast in one line, count per region, growth via slicing (no loop), argmax with unravel_index |
| **B — np.where/select** | B1 Discount Flag | 8 | 3-category flag using np.where or np.select, value_counts correct |
| | B2 Rev Score | 12 | np.percentile for p25/p75, np.select for 3 tiers, value_counts + groupby mean, NRA present |
| **C — Vectorised** | C1 Normalisation | 10 | All 3 transforms in one line each, columns added to df, verification prints min=0/max=1 for scaled |
| | C2 Projection | 10 | reshape(-1,1) broadcast, shape (200,6), mean per month, best month and month 6 total computed |
| **D — Integration** | D1 Outlier + Clip | 10 | IQR bounds computed, np.where flag, np.clip applied, before/after stats printed |
| | D2 Feature Matrix | 10 | 7-col X array, shape/dtype correct, column means+stds, NaN/Inf check = 0, NRA present |
| **★ Bonus** | See below | 10 | — |
| **TOTAL** | | **80 + 10★** | |

---

### ⭐ Bonus Task — Matrix Operations on monthly_sales (10★ pts)

All operations on `monthly_sales` (4×6 array). No Pandas. No loops.

**Part 1 — Normalise each region to its own max (2★):**
```python
region_max = monthly_sales.max(axis=1, keepdims=True)   # shape (4, 1)
normalised = monthly_sales / region_max                   # broadcasting → shape (4, 6)
```
Print `normalised` rounded to 3dp.

**Part 2 — Rank months within each region (4★):**
For each region (row), rank the 6 months from best (1) to worst (6).
Use `np.argsort` — it returns the sorted indices, not the ranks.
Convert argsort output to ranks:

```python
# argsort gives positions of sorted elements (ascending by default)
# argsort of argsort gives ranks
ranks = np.argsort(np.argsort(-monthly_sales, axis=1), axis=1) + 1
```
Print the rank matrix. Which month was consistently ranked #1 across most regions?

**Part 3 — Correlation between regions (4★):**
Compute the correlation matrix between regions using `np.corrcoef(monthly_sales)`.
This gives a 4×4 matrix where entry [i,j] is the correlation between region i and region j.
Print it rounded to 3dp.
Which two regions are most correlated? (Find the off-diagonal maximum.)


In [19]:
# ⭐ BONUS — Matrix Operations on monthly_sales (10★ pts)

# Part 1 – Normalise each region to its own max
region_max = monthly_sales.max(axis=1, keepdims=True)   # shape (4,1)
normalised = monthly_sales / region_max                  # broadcasting
print("Part 1: Normalised (value ÷ region max):")
print(normalised.round(3))

# Part 2 – Rank months within each region (1 = best month)
ranks = np.argsort(np.argsort(-monthly_sales, axis=1), axis=1) + 1
print("\nPart 2: Month ranks (1 = highest sales):")
print(ranks)

# Which month is most often ranked #1?
best_month_counts = (ranks == 1).sum(axis=0)
most_consistent_idx = np.argmax(best_month_counts)
print(f"Month most consistently #1: {month_names[most_consistent_idx]} "
      f"({best_month_counts[most_consistent_idx]} regions)")

# Part 3 – Correlation matrix between regions
corr = np.corrcoef(monthly_sales)   # 4×4 correlation matrix
print("\nPart 3: Correlation matrix (region × region):")
print(corr.round(3))

# Find the most correlated pair (off‑diagonal maximum)
np.fill_diagonal(corr, -np.inf)   # ignore self‑correlation
flat_idx = np.argmax(corr)
r1, r2 = np.unravel_index(flat_idx, corr.shape)
print(f"Most correlated pair: {region_names[r1]} & {region_names[r2]} "
      f"(r = {max(corr[r1,r2], corr[r2,r1]):.3f})")


Part 1: Normalised (value ÷ region max):
[[0.656 0.81  0.624 0.884 0.751 1.   ]
 [0.646 0.829 0.791 0.753 1.    0.905]
 [0.8   0.69  0.819 0.91  0.776 1.   ]
 [0.715 0.818 0.774 0.934 0.839 1.   ]]

Part 2: Month ranks (1 = highest sales):
[[5 3 6 2 4 1]
 [6 3 4 5 1 2]
 [4 6 3 2 5 1]
 [6 4 5 2 3 1]]
Month most consistently #1: Jun (3 regions)

Part 3: Correlation matrix (region × region):
[[1.    0.379 0.632 0.944]
 [0.379 1.    0.056 0.477]
 [0.632 0.056 1.    0.746]
 [0.944 0.477 0.746 1.   ]]
Most correlated pair: North & West (r = 0.944)


---

### Key Takeaway
NumPy is not a separate tool from Pandas — it is the substrate Pandas runs on. Every time you write `.to_numpy()`, you're dropping down a layer to get raw speed. The three operations you'll use most in real ML workflows are: vectorised math (replace every loop you've ever written), `np.where`/`np.select` (replace every apply with a condition), and broadcasting (replace every manual reshape). Master those three and you can preprocess a million-row dataset without a single Python-level loop.

### Interview Frame
> "I use NumPy for anything that needs to run fast at scale. For ML preprocessing I extract Pandas columns to NumPy arrays, apply min-max scaling or z-score standardisation in one vectorised line, flag outliers with np.where based on IQR bounds, and feed the clean matrix directly to Scikit-learn — no loops, no apply, no slowdown."

---
